In [2]:
# imports
# If these fail, please check you're running from an 'activated' 
# environment with (llms) in the command prompt

import os
import json
from icecream import ic
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI
from IPython.display import display, Markdown, update_display

In [3]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [4]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [5]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""


In [6]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [7]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [8]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [9]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [12]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face Brochure

---

## About Hugging Face  
Hugging Face is the vibrant AI community and platform dedicated to building the future of machine learning. It serves as the hub where data scientists, machine learning engineers, researchers, and AI enthusiasts from around the globe collaborate on developing cutting-edge models, datasets, and applications. With over 2 million models, 500,000+ datasets, and 1 million+ applications shared, Hugging Face empowers innovation across modalities including text, image, video, audio, and even 3D.

Whether you are exploring AI apps or diving into an extensive model repository, Hugging Face offers a collaborative space to build, share, and scale AI faster.

---

## Key Features & Offerings

- **Models & Datasets:** Access and contribute to the world’s largest open source collection of machine learning models and datasets.
- **Spaces:** Host and demo AI applications with zero infrastructure management, supporting rich, interactive experiences.
- **Multi-Modality Support:** Work seamlessly across different data types including natural language processing, computer vision, audio, and 3D data.
- **Open Source Ecosystem:** Utilize the Hugging Face open source stack for faster and more efficient ML workflows.
- **Community:** Engage with a global community that fuels collective AI progress through collaboration and knowledge sharing.

---

## Enterprise Solutions

Hugging Face offers tailored enterprise and team solutions to scale AI initiatives securely and efficiently at the organizational level:

- **Team Plan:** Starting at $20/user/month, ideal for fast setup and collaborative work within growing teams.
- **Enterprise Hub:** Flexible contracts with advanced security and governance features including:
  - Single Sign-On (SSO) integration  
  - Data residency and audit logs  
  - Granular access controls and token management  
  - Priority customer support  
  - Increased private storage and ZeroGPU quota boosts  
  - Detailed analytics and billing management

Organizations can effortlessly scale AI development, monitor usage, and ensure compliance with enterprise-grade security policies.

---

## Pricing Overview

- **PRO Account ($9/month):** Elevate personal AI projects with private storage, enhanced inference credits, priority compute resources, and publishing features.
- **Team Plan ($20/user/month):** Designed for collaborative environments with seamless team management, SSO, and enhanced compute resources.
- **Enterprise:** Customizable packages with flexible contracts and dedicated support for large-scale AI deployments.

---

## Company Culture & Community

At its core, Hugging Face fosters an open, collaborative, and inclusive culture that thrives on knowledge sharing and innovation. The community-driven approach is central to the company’s philosophy, providing a welcoming space for contributors of all levels to:

- Build portfolios by sharing work publicly  
- Collaborate on cross-institutional AI projects  
- Contribute to open source tools advancing AI technology  
- Connect with thousands of ML practitioners worldwide

This spirit of collaboration makes Hugging Face more than just a platform—it is a movement shaping the future of artificial intelligence.

---

## Careers & Opportunities

Joining Hugging Face means being part of a passionate team driving open source AI innovation. The company seeks talented individuals across multiple disciplines, including software engineering, research, product, and community engagement roles. Employees enjoy:

- Working at the forefront of AI technologies  
- Access to a vibrant international community  
- Opportunities to influence the future of machine learning tools  
- A flexible, supportive, and mission-driven work environment  

For career opportunities, visit their website to explore current openings and join the AI revolution.

---

## Join Hugging Face

Whether you are an individual developer, part of a growing team, or a forward-thinking enterprise, Hugging Face provides the tools, community, and support to build next-generation AI applications. Discover, create, and scale your AI projects in an ecosystem designed to accelerate machine learning innovation for everyone.

**Explore more at:** [huggingface.co](https://huggingface.co)  

---

**Hugging Face — The AI community building the future.**